# Perception Safety Batch Evaluation - Colab

Use this notebook to run YOLO batch inference on a nuScenes camera subset, compare detections with simple nuScenes annotation counts, and export CSV/Markdown artifacts for the Streamlit app.

In [ ]:
!pip install -q ultralytics opencv-python pandas numpy pillow tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Update this path to match your Google Drive folder.
NUSCENES_ROOT = Path('/content/drive/MyDrive/nuscenes_subset')
METADATA_ROOT = NUSCENES_ROOT / 'v1.0-trainval'
IMAGE_DIR = NUSCENES_ROOT / 'samples' / 'CAM_FRONT'
OUTPUT_DIR = Path('/content/drive/MyDrive/perception_eval_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'yolov8n.pt'
CONFIDENCE_THRESHOLD = 0.25
LOW_CONFIDENCE_THRESHOLD = 0.50
MAX_IMAGES = 500

print('Metadata root:', METADATA_ROOT)
print('Image dir:', IMAGE_DIR)
print('Output dir:', OUTPUT_DIR)
print('Image count:', len(list(IMAGE_DIR.glob('*.jpg'))) if IMAGE_DIR.exists() else 0)

In [ ]:
import json
from collections import Counter, defaultdict

def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def map_nuscenes_category_to_yolo(category_name):
    if category_name.startswith('human.pedestrian'):
        return 'person'
    if category_name == 'vehicle.car':
        return 'car'
    if category_name.startswith('vehicle.truck'):
        return 'truck'
    if category_name.startswith('vehicle.bus'):
        return 'bus'
    if category_name == 'vehicle.motorcycle':
        return 'motorcycle'
    if category_name == 'vehicle.bicycle':
        return 'bicycle'
    if category_name == 'movable_object.trafficcone':
        return 'traffic cone'
    return None

def build_expected_counts_by_sample(metadata_root):
    categories = load_json(metadata_root / 'category.json')
    instances = load_json(metadata_root / 'instance.json')
    annotations = load_json(metadata_root / 'sample_annotation.json')

    category_by_token = {row['token']: row['name'] for row in categories}
    instance_category = {
        row['token']: category_by_token.get(row['category_token'], 'unknown')
        for row in instances
    }

    expected = defaultdict(Counter)
    for ann in annotations:
        category_name = instance_category.get(ann.get('instance_token', ''), 'unknown')
        label = map_nuscenes_category_to_yolo(category_name)
        if label:
            expected[ann['sample_token']][label] += 1
    return {sample: dict(counts) for sample, counts in expected.items()}

def build_sample_token_by_image(metadata_root):
    sample_data = load_json(metadata_root / 'sample_data.json')
    return {
        Path(row['filename']).name: row['sample_token']
        for row in sample_data
        if row.get('filename', '').startswith('samples/CAM_FRONT/') and row.get('is_key_frame')
    }

expected_by_sample = build_expected_counts_by_sample(METADATA_ROOT)
sample_token_by_image = build_sample_token_by_image(METADATA_ROOT)
print('Samples with expected counts:', len(expected_by_sample))
print('CAM_FRONT key frames:', len(sample_token_by_image))

In [ ]:
from ultralytics import YOLO
from tqdm import tqdm
import pandas as pd
import time

model = YOLO(MODEL_NAME)
image_paths = sorted(IMAGE_DIR.glob('*.jpg'))[:MAX_IMAGES]
rows = []
start_time = time.time()

for image_path in tqdm(image_paths):
    results = model.predict(str(image_path), conf=CONFIDENCE_THRESHOLD, verbose=False)
    result = results[0]
    for box in result.boxes:
        cls_id = int(box.cls.item())
        label = result.names[cls_id]
        confidence = float(box.conf.item())
        x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
        rows.append({
            'image': image_path.name,
            'model': MODEL_NAME,
            'confidence_threshold': CONFIDENCE_THRESHOLD,
            'label': label,
            'confidence': confidence,
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,
            'low_confidence': confidence < LOW_CONFIDENCE_THRESHOLD,
        })

elapsed = time.time() - start_time
detections_df = pd.DataFrame(rows)
detections_path = OUTPUT_DIR / 'perception_yolo_results.csv'
detections_df.to_csv(detections_path, index=False)
print('Images evaluated:', len(image_paths))
print('Detections:', len(detections_df))
print('Elapsed seconds:', round(elapsed, 2))
print('Saved:', detections_path)
detections_df.head()

In [ ]:
summary_rows = []

for image_path in image_paths:
    image_name = image_path.name
    sample_token = sample_token_by_image.get(image_name)
    expected = Counter(expected_by_sample.get(sample_token, {}))
    image_detections = detections_df[detections_df['image'] == image_name] if not detections_df.empty else pd.DataFrame()
    detected = Counter(image_detections['label'].tolist()) if not image_detections.empty else Counter()

    labels = set(expected) | set(detected)
    true_positive = sum(min(expected[label], detected[label]) for label in labels)
    missed = {label: expected[label] - detected[label] for label in labels if expected[label] > detected[label]}
    false_pos = {label: detected[label] - expected[label] for label in labels if detected[label] > expected[label]}
    total_detected = sum(detected.values())
    total_expected = sum(expected.values())

    summary_rows.append({
        'image': image_name,
        'sample_token': sample_token,
        'detected_total': total_detected,
        'expected_total': total_expected,
        'missed_total': sum(missed.values()),
        'false_positive_total': sum(false_pos.values()),
        'low_confidence_total': int(image_detections['low_confidence'].sum()) if not image_detections.empty else 0,
        'precision': true_positive / total_detected if total_detected else 0.0,
        'recall': true_positive / total_expected if total_expected else 0.0,
        'expected_counts': dict(expected),
        'detected_counts': dict(detected),
        'missed_objects': missed,
        'false_positives': false_pos,
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / 'perception_eval_summary.csv'
summary_df.to_csv(summary_path, index=False)
print('Saved:', summary_path)
summary_df.head()

In [ ]:
report_path = OUTPUT_DIR / 'perception_failure_report.md'
mean_precision = summary_df['precision'].mean() if not summary_df.empty else 0.0
mean_recall = summary_df['recall'].mean() if not summary_df.empty else 0.0

report = f'''# Perception Batch Evaluation Report

## Setup
- Model: {MODEL_NAME}
- Images evaluated: {len(image_paths)}
- Detection threshold: {CONFIDENCE_THRESHOLD}
- Low-confidence threshold: {LOW_CONFIDENCE_THRESHOLD}

## Aggregate Results
- Total detections: {len(detections_df)}
- Total expected objects: {int(summary_df['expected_total'].sum()) if not summary_df.empty else 0}
- Total missed objects: {int(summary_df['missed_total'].sum()) if not summary_df.empty else 0}
- Total false positives: {int(summary_df['false_positive_total'].sum()) if not summary_df.empty else 0}
- Total low-confidence detections: {int(summary_df['low_confidence_total'].sum()) if not summary_df.empty else 0}
- Mean precision: {mean_precision:.3f}
- Mean recall: {mean_recall:.3f}

## Safety Interpretation
- Missed vulnerable road users or vehicles should be treated as perception safety concerns for SOTIF and ISO 8800 follow-up.
- Low-confidence detections indicate candidates for threshold sensitivity analysis and scenario expansion.
- False positives may cause unnecessary braking or planning disturbances.

## Recommended Follow-up
- Run the same notebook with multiple YOLO models and confidence thresholds.
- Add camera channels beyond CAM_FRONT.
- Move from class-count matching to bounding-box IoU matching.
- Import the CSV files into the Streamlit Project 3 dashboard.
'''

report_path.write_text(report)
print('Saved:', report_path)
print(report)